# Robust Image Preprocessing, Harmonization & Balancing Pipeline for 6-Dataset DR Hybrid Pool

**Author:** Expert Computer Vision & Medical AI Assistant  
**Dataset Scope:** APTOS 2019, EyePACS, Messidor-2, IDRiD, DeepDRiD, DDR (53,977 raw images).  
**Target Outputs:** Dual resolutions (`300x300` & `512x512`), 70/15/15 leak-free stratified splits, CLAHE contrast enhancement, and targeted training set class balancing (exactly 2,000 images/class = 10,000 total train images).



In [1]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import albumentations as A
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from tqdm import tqdm

# Suppress non-critical warnings
warnings.filterwarnings("ignore")

# Random seeds for research reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"[INFO] Initialized preprocessing pipeline with SEED={SEED}")



[INFO] Initialized preprocessing pipeline with SEED=42


## 1. Dynamic Path Resolution & Configuration Dictionary



In [2]:
def get_base_paths() -> tuple[str, str]:
    """
    Dynamically resolves relative paths to `data/raw/` and `data/processed/`
    based on current working directory.
    """
    cwd = os.getcwd()
    raw_dir = os.path.abspath(os.path.join(cwd, "..", "..", "..", "..", "data", "raw"))
    proc_dir = os.path.abspath(os.path.join(cwd, "..", "..", "..", "..", "data", "processed"))
    os.makedirs(proc_dir, exist_ok=True)
    return raw_dir, proc_dir

BASE_RAW_DIR, BASE_PROCESSED_DIR = get_base_paths()
HYBRID_PROCESSED_300 = os.path.join(BASE_PROCESSED_DIR, "hybrid", "300x300")
HYBRID_PROCESSED_512 = os.path.join(BASE_PROCESSED_DIR, "hybrid", "512x512")
HYBRID_RAW_REF = os.path.join(BASE_RAW_DIR, "hybrid")

os.makedirs(HYBRID_PROCESSED_300, exist_ok=True)
os.makedirs(HYBRID_PROCESSED_512, exist_ok=True)
os.makedirs(HYBRID_RAW_REF, exist_ok=True)

print(f"[INFO] Raw Dir: {BASE_RAW_DIR}")
print(f"[INFO] Processed 300x300 Dir: {HYBRID_PROCESSED_300}")
print(f"[INFO] Processed 512x512 Dir: {HYBRID_PROCESSED_512}")

# Raw dataset configuration dictionary
DATASET_CONFIG = {
    "APTOS": {
        "name": "APTOS 2019",
        "csv_path": os.path.join("aptos", "aptos_total.csv"),
        "img_dir": os.path.join("aptos", "aptos_total_images"),
        "id_col": "id_code",
        "label_col": "diagnosis",
        "id_suffix": ".png"
    },
    "DDR": {
        "name": "DDR Dataset",
        "csv_path": os.path.join("ddr", "DR_grading.csv"),
        "img_dir": os.path.join("ddr", "DR_grading", "DR_grading"),
        "id_col": "id_code",
        "label_col": "diagnosis",
        "id_suffix": ".jpg"
    },
    "DeepDRiD": {
        "name": "DeepDRiD",
        "csv_path": os.path.join("deepdrid", "deepdrid_total.csv"),
        "img_dir": os.path.join("deepdrid", "deepdrid_total_images"),
        "id_col": "image_id",
        "label_col": "patient_DR_Level",
        "id_suffix": ".jpg"
    },
    "IDRiD": {
        "name": "IDRiD",
        "csv_path": os.path.join("idrid", "idrid_total.csv"),
        "img_dir": os.path.join("idrid", "idrid_total_images"),
        "id_col": "image_id",
        "label_col": "diagnosis",
        "id_suffix": ".jpg"
    },
    "Messidor-2": {
        "name": "Messidor-2",
        "csv_path": os.path.join("messidor2preprocess", "messidor_data.csv"),
        "img_dir": os.path.join("messidor2preprocess", "messidor-2", "messidor-2", "preprocess"),
        "id_col": "id_code",
        "label_col": "diagnosis",
        "id_suffix": ".png"
    },
    "EyePACS": {
        "name": "EyePACS (Kaggle)",
        "csv_path": os.path.join("zipEyepacs", "trainLabels.csv", "trainLabels.csv"),
        "img_dir": os.path.join("zipEyepacs", "train"),
        "id_col": "image",
        "label_col": "level",
        "id_suffix": ".jpeg"
    }
}

[INFO] Raw Dir: d:\Github\deep-learning-for-computer-vision\data\raw
[INFO] Processed 300x300 Dir: d:\Github\deep-learning-for-computer-vision\data\processed\hybrid\300x300
[INFO] Processed 512x512 Dir: d:\Github\deep-learning-for-computer-vision\data\processed\hybrid\512x512


## 2. Step 1: Master Dataset Harmonization & Schema Standardization

**Clinical Justification:** All 6 datasets map to the International Clinical Diabetic Retinopathy Severity Scale (ICDRS 0-4).
Merging multi-center datasets exposes the network to diverse camera sensors (Topcon, Canon, Zeiss) and illumination conditions, improving cross-domain generalizability.



In [3]:
def parse_patient_and_side(image_id: str, dataset_key: str) -> tuple[str, str]:
    """
    Trích xuất chính xác patient_id và mắt (left/right) từ tên ảnh của từng dataset cụ thể.
    Đảm bảo mắt trái và mắt phải của CÙNG MỘT BỆNH NHÂN luôn có chung một patient_id.
    """
    img_str = str(image_id).lower()
    side = "unknown"
    
    # 1. Nhận diện phía mắt (Left / Right)
    if "left" in img_str or "_l" in img_str:
        side = "left"
    elif "right" in img_str or "_r" in img_str:
        side = "right"
        
    # 2. Trích xuất patient_id tùy thuộc vào từng dataset để gom đúng mắt trái/phải về một người
    key_lower = dataset_key.lower()
    
    if "deepdrid" in key_lower:
        # DeepDRiD: Định dạng '1_l1', '1_r1' -> Cắt lấy phần trước dấu '_' đầu tiên làm ID bệnh nhân ('1')
        patient_id = img_str.split("_")[0] if "_" in img_str else img_str
        
    elif "eyepacs" in key_lower:
        # EyePACS: Định dạng '10_left', '10_right' -> Cắt lấy phần trước dấu '_' ('10')
        patient_id = img_str.split("_")[0] if "_" in img_str else img_str
        
    elif "aptos" in key_lower:
        # APTOS: Thường mỗi ảnh là một bệnh nhân độc lập hoặc mã riêng biệt không có cặp trái/phải rõ ràng
        patient_id = img_str
        
    elif "messidor" in key_lower:
        # Messidor-2: Thường tên ảnh đã là mã bệnh nhân duy nhất
        patient_id = img_str
        
    elif "idrid" in key_lower:
        # IDRiD: Định dạng 'IDRiD_001', 'IDRiD_002' hoặc kèm nhãn mắt
        # Nếu muốn gom chung mắt trái/phải của cùng IDRiD, ta tách bỏ phần hậu tố _left/_right nếu có
        clean_id = img_str.replace("_left", "").replace("_right", "").replace(".jpg", "").replace(".png", "")
        patient_id = clean_id.rsplit("_", 1)[0] if "_" in clean_id else clean_id
        
    elif "ddr" in key_lower:
        # DDR: Tên ảnh chứa mã trung tâm/bệnh nhân
        patient_id = img_str.split("_")[0] if "_" in img_str else img_str
        
    else:
        # Mặc định an toàn: lấy toàn bộ chuỗi làm patient_id độc lập nếu không nhận diện được
        patient_id = img_str
        
    return str(patient_id), side
def load_and_standardize_single_dataset(key: str, cfg: dict, base_dir: str) -> pd.DataFrame:
    full_csv_path = os.path.join(base_dir, cfg["csv_path"])
    full_img_dir = os.path.join(base_dir, cfg["img_dir"])
    
    if not os.path.exists(full_csv_path):
        print(f"[WARNING] Skipping {key} - CSV not found: {full_csv_path}")
        return pd.DataFrame()
        
    raw_df = pd.read_csv(full_csv_path)
    label_col = cfg["label_col"]
    if label_col not in raw_df.columns:
        possible = [c for c in raw_df.columns if "dr" in c.lower() or "level" in c.lower() or "diagnosis" in c.lower()]
        if possible:
            label_col = possible[0]
            
    df = raw_df[[cfg["id_col"], label_col]].copy()
    df.columns = ["image_id", "diagnosis"]
    df["dataset_name"] = cfg["name"]
    
    # Ensure correct suffix
    if cfg["id_suffix"]:
        df["image_file"] = df["image_id"].astype(str).apply(lambda x: x if x.endswith(cfg["id_suffix"]) else x + cfg["id_suffix"])
    else:
        df["image_file"] = df["image_id"].astype(str)
        
    df["raw_full_path"] = df["image_file"].apply(lambda f: os.path.join(full_img_dir, f))
    
    # Clean non-integer labels
    df["diagnosis"] = pd.to_numeric(df["diagnosis"], errors="coerce")
    df = df.dropna(subset=["diagnosis"]).copy()
    df["diagnosis"] = df["diagnosis"].astype(int)
    df = df[df["diagnosis"].isin([0, 1, 2, 3, 4])]
    # GỌI HÀM PARSE ĐỂ TẠO PATIENT_ID CHỐNG LEAKAGE
    parsed_metadata = df["image_id"].apply(lambda x: parse_patient_and_side(x, dataset_key=key))
    df["patient_id"] = [p[0] for p in parsed_metadata]
    df["side"] = [p[1] for p in parsed_metadata]
    return df

def merge_and_harmonize_all_datasets(config_dict: dict, base_dir: str) -> pd.DataFrame:
    dfs = []
    for key, cfg in config_dict.items():
        df_sub = load_and_standardize_single_dataset(key, cfg, base_dir)
        if not df_sub.empty:
            dfs.append(df_sub)
            print(f"[HARMONIZED] {cfg['name']}: {len(df_sub):,} records")
            
    master_df = pd.concat(dfs, ignore_index=True)
    print(f"\n[SUCCESS] Master Harmonized Dataset: {len(master_df):,} total images across {master_df['dataset_name'].nunique()} sources.")
    return master_df

master_df = merge_and_harmonize_all_datasets(DATASET_CONFIG, BASE_RAW_DIR)

[HARMONIZED] APTOS 2019: 3,662 records
[HARMONIZED] DDR Dataset: 12,522 records
[HARMONIZED] DeepDRiD: 2,000 records
[HARMONIZED] IDRiD: 516 records
[HARMONIZED] Messidor-2: 1,744 records
[HARMONIZED] EyePACS (Kaggle): 35,126 records

[SUCCESS] Master Harmonized Dataset: 55,570 total images across 6 sources.


In [4]:
master_df.head()

,image_id,diagnosis,dataset_name,image_file,raw_full_path,patient_id,side
0,1ae8c165fd53,2,APTOS 2019,1ae8c165fd53.png,d:\Github\deep-learning-for-computer-vision\da...,1ae8c165fd53,unknown
1,1b329a127307,1,APTOS 2019,1b329a127307.png,d:\Github\deep-learning-for-computer-vision\da...,1b329a127307,unknown
2,1b32e1d775ea,4,APTOS 2019,1b32e1d775ea.png,d:\Github\deep-learning-for-computer-vision\da...,1b32e1d775ea,unknown
3,1b3647865779,0,APTOS 2019,1b3647865779.png,d:\Github\deep-learning-for-computer-vision\da...,1b3647865779,unknown
4,1b398c0494d1,0,APTOS 2019,1b398c0494d1.png,d:\Github\deep-learning-for-computer-vision\da...,1b398c0494d1,unknown


## 4. Step 2 & Step 3: Geometric Preprocessing & Image Enhancement

**Auto-Crop ROI:** Thresholding technique to remove non-diagnostic black borders around the fundus circular region.  


In [5]:
def crop_fundus_roi(img_bgr: np.ndarray, tolerance: int = 7) -> np.ndarray:
    """
    Auto-crops uninformative black borders around fundus photograph.
    """
    if img_bgr is None or img_bgr.size == 0:
        return img_bgr
        
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mask = gray > tolerance
    
    if not np.any(mask):
        return img_bgr
        
    # Crop to bounding box of non-black pixels
    row_mask = np.any(mask, axis=1)
    col_mask = np.any(mask, axis=0)
    ymin, ymax = np.where(row_mask)[0][[0, -1]]
    xmin, xmax = np.where(col_mask)[0][[0, -1]]
    
    cropped = img_bgr[ymin:ymax+1, xmin:xmax+1]
    return cropped if cropped.size > 0 else img_bgr

# def enhance_fundus_image(img_rgb: np.ndarray) -> np.ndarray:
#     """
#     Extracts Green channel weighted blending, applies CLAHE, and Gaussian Blur.
#     """
#     # Convert RGB to BGR for OpenCV processing
#     bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    
#     # 1. Auto-Crop ROI
#     cropped_bgr = crop_fundus_roi(bgr)
    
#     # 2. Green Channel weighted blend
#     b, g, r = cv2.split(cropped_bgr)
#     # Blend green channel heavily with subtle gray balance
#     green_blended = cv2.addWeighted(g, 0.75, cv2.cvtColor(cropped_bgr, cv2.COLOR_BGR2GRAY), 0.25, 0)
    
#     # 3. CLAHE Contrast Enhancement
#     clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
#     enhanced_g = clahe.apply(green_blended)
    
#     # Reconstruct 3-channel RGB image (using enhanced green for all channels or merging)
#     enhanced_bgr = cv2.merge([b, enhanced_g, r])
    
#     # 4. Gaussian Blur Noise Reduction
#     blurred_bgr = cv2.GaussianBlur(enhanced_bgr, (3, 3), 0)
    
#     return cv2.cvtColor(blurred_bgr, cv2.COLOR_BGR2RGB)
def base_geometric_preprocess(img_rgb: np.ndarray) -> np.ndarray:
    """
    Chỉ thực hiện Crop ROI và giữ nguyên vẹn dải màu RGB tự nhiên của ảnh gốc.
    Không ép buộc CLAHE hay Ben Graham ở đây để nhường chỗ cho Ablation Study sau này.
    """
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    cropped_bgr = crop_fundus_roi(bgr)
    return cv2.cvtColor(cropped_bgr, cv2.COLOR_BGR2RGB)


## 5. Step 4: Albumentations Augmentation & Targeted Training Set Balancing

**Balancing Strategy (Applied STRICTLY to 70% Training Set):**
- Target Count: Exactly 2,000 images per DR severity grade (10,000 total training images).
- Class 0, Class 1, Class 2: Randomly undersampled to 2,000 images.
- Class 3, Class 4: Oversampled & augmented via Albumentations to 2,000 images.
- Validation (8,097) & Test (8,097) sets: Left untouched in raw distribution.



In [6]:
# from torch._export.passes import add_runtime_assertions_for_constraints_pass
# from torch._export.passes import add_runtime_assertions_for_constraints_pass
# from torch._export.passes import add_runtime_assertions_for_constraints_pass
# train_augmentor = A.Compose([
#     A.HorizontalFlip(p=0.5),
#     A.VerticalFlip(p=0.5),
#     A.RandomRotate90(p=0.5),
#     A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.5),
#     A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
# ])

# def balance_and_preprocess_split(
#     split_name: str, 
#     df: pd.DataFrame, 
#     target_per_class: int = 2000, 
#     is_train: bool = False
# ) -> pd.DataFrame:
    
#     records = []
    
#     for grade in range(5):
#         grade_subset = df[df["diagnosis"] == grade].copy()
#         # Filter for rows whose raw image file actually exists on disk
#         valid_mask = grade_subset["raw_full_path"].apply(lambda p: os.path.isfile(p))
#         grade_subset_existing = grade_subset[valid_mask].reset_index(drop=True)
#         current_count = len(grade_subset_existing)
        
#         if current_count == 0:
#             print(f"[WARNING] No existing image files found for grade {grade} in split '{split_name}'.")
#             continue

#         if is_train:
#             if current_count >= target_per_class:
#                 selected_subset = grade_subset_existing.sample(target_per_class, random_state=SEED).reset_index(drop=True)
#                 aug_multiplier = 1
#             else:
#                 selected_subset = grade_subset_existing.copy()
#                 aug_multiplier = (target_per_class // current_count) + 1
#         else:
#             selected_subset = grade_subset_existing.copy()
#             aug_multiplier = 1
            
#         count_saved = 0
        
#         # Loop until target_per_class is reached for train set
#         while is_train and count_saved < target_per_class:
#             for idx, row in selected_subset.iterrows():
#                 if count_saved >= target_per_class:
#                     break
                
#                 img_path = row["raw_full_path"]
#                 try:
#                     raw_img = Image.open(img_path).convert("RGB")
#                     img_np = np.array(raw_img)
#                 except Exception:
#                     continue

#                 if count_saved < current_count and not is_train:
#                     processed_np = enhance_fundus_image(img_np)
#                     is_aug = False
#                 elif count_saved < len(selected_subset) and current_count >= target_per_class:
#                     processed_np = enhance_fundus_image(img_np)
#                     is_aug = False
#                 else:
#                     enhanced = enhance_fundus_image(img_np)
#                     augmented = train_augmentor(image=enhanced)
#                     processed_np = augmented["image"]
#                     is_aug = True

#                 out_name = f"{split_name}_g{grade}_{row['image_id']}_c{count_saved}.png"
#                 out_name_clean = re.sub(r'[^\w\.-]', '_', out_name)
                
#                 path_300 = os.path.join(HYBRID_PROCESSED_300, split_name, str(grade), out_name_clean)
#                 path_512 = os.path.join(HYBRID_PROCESSED_512, split_name, str(grade), out_name_clean)
                
#                 os.makedirs(os.path.dirname(path_300), exist_ok=True)
#                 os.makedirs(os.path.dirname(path_512), exist_ok=True)
                
#                 img_pil = Image.fromarray(processed_np)
#                 img_pil.resize((300, 300), Image.Resampling.BILINEAR).save(path_300)
#                 img_pil.resize((512, 512), Image.Resampling.BILINEAR).save(path_512)
                
#                 records.append({
#                     "split": split_name,
#                     "diagnosis": grade,
#                     "image_id": row["image_id"],
#                     "dataset_name": row["dataset_name"],
#                     "path_300": path_300,
#                     "path_512": path_512,
#                     "is_augmented": is_aug
#                 })
                
#                 count_saved += 1

#         if not is_train:
#             for idx, row in selected_subset.iterrows():
#                 img_path = row["raw_full_path"]
#                 try:
#                     raw_img = Image.open(img_path).convert("RGB")
#                     img_np = np.array(raw_img)
#                 except Exception:
#                     continue

#                 processed_np = enhance_fundus_image(img_np)
#                 out_name = f"{split_name}_g{grade}_{row['image_id']}.png"
#                 out_name_clean = re.sub(r'[^\w\.-]', '_', out_name)
                
#                 path_300 = os.path.join(HYBRID_PROCESSED_300, split_name, str(grade), out_name_clean)
#                 path_512 = os.path.join(HYBRID_PROCESSED_512, split_name, str(grade), out_name_clean)
                
#                 os.makedirs(os.path.dirname(path_300), exist_ok=True)
#                 os.makedirs(os.path.dirname(path_512), exist_ok=True)
                
#                 img_pil = Image.fromarray(processed_np)
#                 img_pil.resize((300, 300), Image.Resampling.BILINEAR).save(path_300)
#                 img_pil.resize((512, 512), Image.Resampling.BILINEAR).save(path_512)
                
#                 records.append({
#                     "split": split_name,
#                     "diagnosis": grade,
#                     "image_id": row["image_id"],
#                     "dataset_name": row["dataset_name"],
#                     "path_300": path_300,
#                     "path_512": path_512,
#                     "is_augmented": False
#                 })
                
#                 count_saved += 1
#                 if is_train and count_saved >= target_per_class:
#                     break
                    
#     return pd.DataFrame(records)

# # Execute preprocessing & balancing on Train, Val, and Test splits
# print("\n=== PROCESSING 70% BALANCED TRAIN SET (TARGET: 2,000 PER CLASS) ===")
# train_processed_df = balance_and_preprocess_split("train", train_df_raw, target_per_class=2000, is_train=True)

# print("\n=== PROCESSING 15% UNTOUCHED VALIDATION SET ===")
# val_processed_df = balance_and_preprocess_split("val", val_df_raw, is_train=False)

# print("\n=== PROCESSING 15% UNTOUCHED TEST SET ===")
# test_processed_df = balance_and_preprocess_split("test", test_df_raw, is_train=False)

# ==========================================
# 2. LẤY MẪU ĐẢM BẢO TỪNG DATASET KHÔNG BỊ BỎ SÓT (GUARANTEED MULTI-SOURCE SAMPLING)
# ==========================================
def create_balanced_master_pool(master_df: pd.DataFrame, target_range=(1500, 2600), seed=42) -> pd.DataFrame:
    """
    Phân bổ số lượng ảnh đồng đều giữa các dataset trong cùng một lớp,
    tránh việc dataset lớn (EyePACS) nuốt chửng các dataset nhỏ (IDRiD, Messidor).
    """
    balanced_dfs = []
    np.random.seed(seed)
    
    print("\n" + "="*70)
    print("=== LẤY MẪU CÂN BẰNG ĐỒNG ĐỀU CÁC DATASET TRONG MASTER HYBRID POOL ===")
    print("="*70)
    
    for grade in range(5):
        grade_subset = master_df[master_df["diagnosis"] == grade].copy()
        valid_mask = grade_subset["raw_full_path"].apply(lambda p: os.path.isfile(p))
        grade_subset_existing = grade_subset[valid_mask].reset_index(drop=True)
        current_count = len(grade_subset_existing)
        
        if current_count == 0:
            continue
            
        print(f"\n--- [DIAGNOSIS GRADE {grade}] (Tổng gốc có sẵn: {current_count:,} ảnh) ---")
        
        datasets_in_grade = grade_subset_existing["dataset_name"].unique()
        num_datasets = len(datasets_in_grade)
        
        # Nếu tổng số ảnh vượt ngưỡng tối đa, ta tiến hành cân bằng nguồn
        if current_count > target_range[1]:
            target_count = int(np.random.RandomState(seed + grade).randint(target_range[0], target_range[1] + 1))
            print(f"👉 Mục tiêu chọn mẫu cho Grade {grade}: {target_count:,} ảnh (chia đều cho {num_datasets} nguồn dataset)")
            
            # Tính hạn mức cơ sở ban đầu cho mỗi dataset
            base_quota = target_count // num_datasets
            
            sampled_datasets = []
            leftover_quota = 0
            eligible_datasets = []
            
            # Vòng 1: Cấp quota cơ sở và xử lý các dataset không đủ số lượng
            for ds_name in datasets_in_grade:
                ds_subset = grade_subset_existing[grade_subset_existing["dataset_name"] == ds_name]
                ds_count = len(ds_subset)
                
                if ds_count <= base_quota:
                    # Nếu dataset có ít hơn quota cơ sở (vd IDRiD), lấy sạch 100% số ảnh nó có
                    sampled_datasets.append(ds_subset)
                    leftover_quota += (base_quota - ds_count) # Phần dư ra nhường cho các dataset lớn
                    print(f"   • {ds_name} (Nhỏ): Lấy toàn bộ {ds_count:,} / {ds_count:,} ảnh (Dư quota: {base_quota - ds_count})")
                else:
                    eligible_datasets.append((ds_name, ds_subset, ds_count))
            
            # Vòng 2: Phân bổ lại phần quota dư cho các dataset lớn hơn
            if eligible_datasets:
                quota_per_large_ds = base_quota + (leftover_quota // len(eligible_datasets))
                for ds_name, ds_subset, ds_count in eligible_datasets:
                    # Chặn trần ở mức quota mới tính toán
                    take_count = min(quota_per_large_ds, ds_count)
                    ds_sampled = ds_subset.sample(take_count, random_state=seed)
                    sampled_datasets.append(ds_sampled)
                    print(f"   • {ds_name} (Lớn): Đã chọn {take_count:,} / {ds_count:,} ảnh")
            
            selected_subset = pd.concat(sampled_datasets, ignore_index=True)
            
            # Tinh chỉnh nhẹ lần cuối nếu tổng số lượng bị lệch đôi chút do làm tròn
            if len(selected_subset) > target_count:
                selected_subset = selected_subset.sample(target_count, random_state=seed).reset_index(drop=True)
            elif len(selected_subset) < target_count:
                remaining = grade_subset_existing[~grade_subset_existing.index.isin(selected_subset.index)]
                if not remaining.empty:
                    diff = target_count - len(selected_subset)
                    extra = remaining.sample(min(diff, len(remaining)), random_state=seed)
                    selected_subset = pd.concat([selected_subset, extra], ignore_index=True)
                    
            print(f"   ↳ Tổng thực tế sau khi cân bằng nguồn: {len(selected_subset):,} ảnh")
        else:
            # Nếu tổng số ảnh của lớp ít hơn ngưỡng, giữ nguyên toàn bộ
            selected_subset = grade_subset_existing.copy()
            print(f"👉 Lớp thiểu số (< {target_range[0]} ảnh), giữ nguyên toàn bộ {current_count:,} ảnh từ các nguồn:")
            for ds_name in datasets_in_grade:
                cnt = len(grade_subset_existing[grade_subset_existing["dataset_name"] == ds_name])
                print(f"   • {ds_name}: {cnt:,} ảnh")
            
        balanced_dfs.append(selected_subset)
        
    master_balanced_df = pd.concat(balanced_dfs, ignore_index=True)
    print("\n" + "="*70)
    print(f"[SUCCESS] Tổng số ảnh trong Master Balanced Pool: {len(master_balanced_df):,}")
    print("="*70)
    return master_balanced_df

def split_master_pool_patient_level(master_balanced_df: pd.DataFrame, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_state=42):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-5, "Ratios must sum to 1.0"
    
    temp_ratio = val_ratio + test_ratio
    gss_train = GroupShuffleSplit(n_splits=1, train_size=train_ratio, random_state=random_state)
    train_idx, temp_idx = next(gss_train.split(master_balanced_df, groups=master_balanced_df['patient_id']))
    
    train_df = master_balanced_df.iloc[train_idx].copy()
    temp_df = master_balanced_df.iloc[temp_idx].copy()
    
    gss_test = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=random_state)
    val_idx, test_idx = next(gss_test.split(temp_df, groups=temp_df['patient_id']))
    
    val_df = temp_df.iloc[val_idx].copy()
    test_df = temp_df.iloc[test_idx].copy()
    
    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"
    
    print("\n=== KẾT QUẢ CHIA TỶ LỆ 70/15/15 (CHỐNG LEAKAGE TUYỆT ĐỐI) ===")
    print(f"Train Set (70%): {len(train_df):,} ảnh | Bệnh nhân: {train_df['patient_id'].nunique():,}")
    print(f"Val Set   (15%): {len(val_df):,} ảnh | Bệnh nhân: {val_df['patient_id'].nunique():,}")
    print(f"Test Set  (15%): {len(test_df):,} ảnh | Bệnh nhân: {test_df['patient_id'].nunique():,}")
    
    return train_df, val_df, test_df


# ==========================================
# 4. HÀM LƯU ẢNH GỐC (NATIVE RESOLUTION) KHÔNG RESIZE, KHÔNG PADDING
# ==========================================
NATIVE_PROCESSED_DIR = os.path.join(BASE_PROCESSED_DIR, "hybrid", "native_cropped")
os.makedirs(NATIVE_PROCESSED_DIR, exist_ok=True)

def process_and_save_native_resolution(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    records = []
    print(f"\n⏳ Đang cắt viền ROI và lưu ảnh giữ nguyên độ phân giải gốc cho tập [{split_name.upper()}]...")
    
    for idx, row in df.iterrows():
        img_path = row["raw_full_path"]
        try:
            raw_img = Image.open(img_path).convert("RGB")
            img_np = np.array(raw_img)
        except Exception:
            continue

        # Chỉ gọi hàm base_geometric_preprocess (Crop viền ROI, giữ nguyên native resolution và màu sắc)
        processed_rgb = base_geometric_preprocess(img_np)

        out_name = f"{split_name}_g{row['diagnosis']}_{str(row['image_id']).replace('/', '_')}.png"
        out_name_clean = re.sub(r'[^\w\.-]', '_', out_name)
        
        save_dir = os.path.join(NATIVE_PROCESSED_DIR, split_name, str(row['diagnosis']))
        os.makedirs(save_dir, exist_ok=True)
        
        native_path = os.path.join(save_dir, out_name_clean)
        Image.fromarray(processed_rgb).save(native_path)
        
        record = row.to_dict()
        record["native_path"] = native_path  # Lưu đường dẫn ảnh gốc phục vụ cho nhánh Local MIL
        records.append(record)
        
    return pd.DataFrame(records)


# ==========================================
# 5. THỰC THI TOÀN BỘ QUY TRÌNH
# ==========================================
master_balanced_df = create_balanced_master_pool(master_df, target_range=(1500, 2500), seed=SEED)
train_raw_df, val_raw_df, test_raw_df = split_master_pool_patient_level(master_balanced_df, random_state=SEED)

train_processed_df = process_and_save_native_resolution(train_raw_df, "train")
val_processed_df = process_and_save_native_resolution(val_raw_df, "val")
test_processed_df = process_and_save_native_resolution(test_raw_df, "test")

train_processed_df.to_csv(os.path.join(BASE_PROCESSED_DIR, "hybrid", "train_metadata.csv"), index=False)
val_processed_df.to_csv(os.path.join(BASE_PROCESSED_DIR, "hybrid", "val_metadata.csv"), index=False)
test_processed_df.to_csv(os.path.join(BASE_PROCESSED_DIR, "hybrid", "test_metadata.csv"), index=False)

print("\n" + "="*50)
print("=== HOÀN TẤT PIPELINE (ĐẢM BẢO ĐỦ NGUỒN & GIỮ NATIVE RESOLUTION) ===")
print("="*50)
print("Train Set Distribution:\n", train_processed_df["diagnosis"].value_counts().sort_index())
print("\nVal Set Distribution:\n", val_processed_df["diagnosis"].value_counts().sort_index())
print("\nTest Set Distribution:\n", test_processed_df["diagnosis"].value_counts().sort_index())


=== LẤY MẪU CÂN BẰNG ĐỒNG ĐỀU CÁC DATASET TRONG MASTER HYBRID POOL ===

--- [DIAGNOSIS GRADE 0] (Tổng gốc có sẵn: 35,197 ảnh) ---
👉 Mục tiêu chọn mẫu cho Grade 0: 1,602 ảnh (chia đều cho 6 nguồn dataset)
   • IDRiD (Nhỏ): Lấy toàn bộ 168 / 168 ảnh (Dư quota: 99)
   • APTOS 2019 (Lớn): Đã chọn 286 / 1,805 ảnh
   • DDR Dataset (Lớn): Đã chọn 286 / 6,266 ảnh
   • DeepDRiD (Lớn): Đã chọn 286 / 680 ảnh
   • Messidor-2 (Lớn): Đã chọn 286 / 468 ảnh
   • EyePACS (Kaggle) (Lớn): Đã chọn 286 / 25,810 ảnh
   ↳ Tổng thực tế sau khi cân bằng nguồn: 1,602 ảnh

--- [DIAGNOSIS GRADE 1] (Tổng gốc có sẵn: 4,031 ảnh) ---
👉 Mục tiêu chọn mẫu cho Grade 1: 2,336 ảnh (chia đều cho 6 nguồn dataset)
   • APTOS 2019 (Nhỏ): Lấy toàn bộ 370 / 370 ảnh (Dư quota: 19)
   • DeepDRiD (Nhỏ): Lấy toàn bộ 356 / 356 ảnh (Dư quota: 33)
   • IDRiD (Nhỏ): Lấy toàn bộ 25 / 25 ảnh (Dư quota: 364)
   • Messidor-2 (Nhỏ): Lấy toàn bộ 207 / 207 ảnh (Dư quota: 182)
   • DDR Dataset (Lớn): Đã chọn 630 / 630 ảnh
   • EyePACS (Kaggle

## 6. Step 6: Pixel Normalization & Pipeline Verification



In [7]:
def normalize_image_tensor(img_np: np.ndarray, mode: str = "zscore") -> np.ndarray:
    """
    Normalizes pixel values to [0, 1] or standard ImageNet Z-score.
    """
    img_float = img_np.astype(np.float32) / 255.0
    if mode == "minmax":
        return img_float
    elif mode == "zscore":
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        return (img_float - mean) / std
    return img_float

# Save processed metadata tables
train_processed_df.to_csv(os.path.join(BASE_PROCESSED_DIR, "hybrid", "train_balanced_metadata.csv"), index=False)
val_processed_df.to_csv(os.path.join(BASE_PROCESSED_DIR, "hybrid", "val_raw_metadata.csv"), index=False)
test_processed_df.to_csv(os.path.join(BASE_PROCESSED_DIR, "hybrid", "test_raw_metadata.csv"), index=False)

print("\n=== FINAL PREPROCESSING & BALANCING SUMMARY ===")
print("Balanced Train Set (300x300 & 512x512):")
print(train_processed_df["diagnosis"].value_counts().sort_index())

print("\nUntouched Validation Set:")
print(val_processed_df["diagnosis"].value_counts().sort_index())

print("\nUntouched Test Set:")
print(test_processed_df["diagnosis"].value_counts().sort_index())



=== FINAL PREPROCESSING & BALANCING SUMMARY ===
Balanced Train Set (300x300 & 512x512):
diagnosis
0    1084
1    1590
2    1493
3    1252
4    1504
Name: count, dtype: int64

Untouched Validation Set:
diagnosis
0    330
1    400
2    470
3    342
4    346
Name: count, dtype: int64

Untouched Test Set:
diagnosis
0    188
1    346
2    325
3    264
4    329
Name: count, dtype: int64
